In [1]:
import os
%pwd


'/home/harris/pers/Kidney-classification/research'

In [2]:
os.chdir("../")
%pwd

'/home/harris/pers/Kidney-classification'

In [3]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class DataIngestionConfig:
    """
    Data Ingestion Configuration
    """
    root_dir: Path
    source_url: str
    local_data_file: Path
    unzip_dir: Path

In [4]:
from CNNClassifier.constants import *
from CNNClassifier.utils.common import read_yaml, create_directories

In [5]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath=CONFIG_FILE_PATH,
        params_filepath=PARAMS_FILE_PATH):
        
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])
        
    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion
        create_directories([config.root_dir])
        
        data_ingestion_config = DataIngestionConfig(
            root_dir=config.root_dir,
            source_url=config.source_url,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir
        )
        
        return data_ingestion_config

In [6]:
import os
import zipfile
import gdown
from CNNClassifier import logger
from CNNClassifier.utils.common import get_size


class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config
 
    def download_file(self) -> str:
        """
        Download file from Google Drive
        """
        try:
            dataset_url = self.config.source_url
            zip_down_dir = self.config.local_data_file
            os.makedirs("artifacts/data_ingestion", exist_ok=True)
            logger.info(f"Downloading data from{dataset_url} into {zip_down_dir}")
            
            file_id = dataset_url.split("/")[-2]
            prefix = 'https://drive.google.com/uc?id='
            gdown.download(f"{prefix}{file_id}", zip_down_dir, quiet=False)
            
            logger.info(f"Downloaded data from {dataset_url} into {zip_down_dir}")
            
        except Exception as e:
            logger.exception(e)
            raise e
        
    def extract_zip_file(self):
        """
        Extract zip file
        """
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_path)
            logger.info(f"Extracted file to {unzip_path}")
            

In [7]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
except Exception as e:
    logger.exception(e)
    raise e

[2025-04-30 03:03:03,714: INFO: common]: YAML file config/config.yaml loaded successfully.
[2025-04-30 03:03:03,719: INFO: common]: YAML file params.yaml loaded successfully.
[2025-04-30 03:03:03,720: INFO: common]: Directory created successfully at: artifacts
[2025-04-30 03:03:03,722: INFO: common]: Directory created successfully at: artifacts/data_ingestion
[2025-04-30 03:03:03,723: INFO: 148512004]: Downloading data fromhttps://drive.google.com/file/d/1ebccsba4Euw3441nV2rPvaxa5KWJpaQ7/view?usp=sharing into artifacts/data_ingestion/data.zip


Downloading...
From (original): https://drive.google.com/uc?id=1ebccsba4Euw3441nV2rPvaxa5KWJpaQ7
From (redirected): https://drive.google.com/uc?id=1ebccsba4Euw3441nV2rPvaxa5KWJpaQ7&confirm=t&uuid=408a32cb-95f7-4193-8fdd-f56a0d3c2fd6
To: /home/harris/pers/Kidney-classification/artifacts/data_ingestion/data.zip
  7%|▋         | 66.1M/940M [00:24<05:26, 2.68MB/s]

KeyboardInterrupt: 

  7%|▋         | 66.1M/940M [00:39<05:26, 2.68MB/s]